# RAG & Knowledge Systems: Agentic RAG Patterns

In a standard RAG pipeline, the flow is linear: Query -> Retrieve -> Generate. In **Agentic RAG**, the agent takes the driver's seat. It can decide *if* it needs to search, evaluate the *quality* of what it found, and try again if the results are poor. We use **Hugging Face** for local embeddings.

## 1. The Autonomous Advantage
- **Evaluation**: "Is this document relevant to the user's question?"
- **Routing**: "Should I search the documentation, the news, or the user's internal emails?"
- **Self-Correction**: "The search failed to answer the question. I need to rephrase and try again."

---

## 2. Environment Setup
We will integrate our local retrievers into an Agentic loop using **Gemini 2.5 Flash**.

In [1]:
import os
from typing import List, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# Create sample Knowledge Bases for HR and IT
docs_hr = [Document(page_content="Vacation policy: All employees get 20 days per year.", metadata={"dept": "HR"})]
docs_it = [Document(page_content="Password reset: Visit the dashboard at sso.company.com.", metadata={"dept": "IT"})]

db_hr = Chroma.from_documents(docs_hr, embeddings)
db_it = Chroma.from_documents(docs_it, embeddings)

print("Agentic RAG environment ready with Local Embeddings!")

/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:234: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/balamurugan/Documents/bala_github/AgenticAI-LearningPath/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with cri

Agentic RAG environment ready with Local Embeddings!


## 3. Pattern 1: RAG as a Tool
The simplest agentic pattern is to wrap the retriever as a tool. The agent only uses it if the question requires external data.

In [2]:
@tool
def search_hr_docs(query: str):
    """Useful for answering questions about human resources, leave, or company policies."""
    docs = db_hr.similarity_search(query, k=1)
    return docs[0].page_content

@tool
def search_it_portal(query: str):
    """Useful for technical support, passwords, and hardware issues."""
    docs = db_it.similarity_search(query, k=1)
    return docs[0].page_content

tools = [search_hr_docs, search_it_portal]
llm_with_tools = llm.bind_tools(tools)

query = "How do I reset my credentials?"
response = llm_with_tools.invoke(query)

print(f"User Query: {query}")
for tool_call in response.tool_calls:
    print(f"Agent Decision: Use tool '{tool_call['name']}' with args {tool_call['args']}")

User Query: How do I reset my credentials?
Agent Decision: Use tool 'search_it_portal' with args {'query': 'reset credentials'}


## 4. Pattern 2: Router-Based RAG
Before retrieving, we use a structured LLM call to decide *which* knowledge base to search. This provides higher precision than searching everything at once.

In [3]:
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""
    datasource: Literal["hr_manual", "it_portal"] = Field(
        ...,
        description="Given a user question, choose whether to route it to hr_manual or it_portal.",
    )

structured_llm_router = llm.with_structured_output(RouteQuery)

questions = ["How many holiday days do I have?", "My mouse is not working."]
for q in questions:
    decision = structured_llm_router.invoke(q)
    print(f"Question: '{q}' -> Route to: {decision.datasource}")

Question: 'How many holiday days do I have?' -> Route to: hr_manual
Question: 'My mouse is not working.' -> Route to: it_portal


## 5. Pattern 3: Self-Evaluation (Corrective RAG)
The agent retrieves documents, then uses a 'Grader' to check if they are actually relevant. If they are 'poor,' the agent can decide to search elsewhere or ask for clarification.

In [4]:
class GradeDocument(BaseModel):
    """Binary score for relevance check on retrieved documents."""
    relevance: Literal["yes", "no"] = Field(description="Documents are relevant to the question, 'yes' or 'no'")

def corrective_rag_demo(query):
    # 1. Retrieve
    docs = db_hr.similarity_search(query, k=1)
    doc_content = docs[0].page_content
    
    # 2. Grade
    grader_llm = llm.with_structured_output(GradeDocument)
    grade = grader_llm.invoke(f"Query: {query} \nDocument: {doc_content}")
    
    print(f"Query: {query}")
    print(f"Retrieved: {doc_content}")
    
    if grade.relevance == "yes":
        print("Decision: Document is relevant. Proceeding to generation.")
    else:
        print("Decision: Document IRRELEVANT. Triggering fallback search...")
        # In a real agent, this would trigger a web search tool or a generic search.
        print("Fallback Result: [Simulated Web Search Result] No relevant HR info found.")

corrective_rag_demo("What is the weather in Paris?")

Query: What is the weather in Paris?
Retrieved: Vacation policy: All employees get 20 days per year.
Decision: Document IRRELEVANT. Triggering fallback search...
Fallback Result: [Simulated Web Search Result] No relevant HR info found.


## 6. Pattern 4: Hallucination Grader
After generating an answer, the agent checks if the answer is grounded in the context. If the answer contains "made up" facts, it's flagged as a hallucination.

In [5]:
class GroundingCheck(BaseModel):
    """Binary score for hallucination check."""
    is_grounded: bool = Field(description="True if the answer contains ONLY facts found in the context")

def check_hallucination(context, answer):
    checker = llm.with_structured_output(GroundingCheck)
    res = checker.invoke(f"Context: {context} \nAnswer: {answer}")
    return res.is_grounded

context = "The capital of France is Paris."
good_answer = "Paris is France's capital city."
hallucinated_answer = "The capital of France is Paris and it has 50 districts."

print(f"Is 'Good Answer' grounded? {check_hallucination(context, good_answer)}")
print(f"Is 'Bad Answer' grounded? {check_hallucination(context, hallucinated_answer)}")

Is 'Good Answer' grounded? True
Is 'Bad Answer' grounded? False


## 7. Multi-Step Reasoner (Self-RAG Loop)
For complex questions, the agent can perform multiple search steps. For example: "Who is the CEO and what is their vacation policy?"
1. Search: "Who is the CEO?" -> Result: "Alice."
2. Search: "Alice vacation policy" -> Result: "CEO has unlimited leave."

---

## 8. LangGraph for Structured Loops
While we are using manual loops here, **LangGraph** is the industry-standard way to manage these states. It allows you to build complex RAG cycles as a graph:
1. **NODE 1**: Retrieve
2. **NODE 2**: Grade Docs (EDGE: If bad -> NODE 1.5: Re-write Query)
3. **NODE 3**: Generate
4. **NODE 4**: Grade Answer (EDGE: If Hallucination -> NODE 3)

We will explore this deeply in the next module!

---

## 9. Local vs Cloud in Agentic RAG
In an Agentic world, you make **many** small calls to the LLM (for grading, routing, rewriting). 
- **Local Embeddings (Hugging Face)**: Crucial for keeping speed high during these inner loops.
- **Small LLMs**: You can use smaller local models (like Llama 3) for the routing/grading tasks to save costs, and only use Gemini for the final creative response.

---

## 10. Summary & Pro-Tips
1. **Never Trust, Always Verify**: Never use a retrieved document without grading it for relevance first.
2. **Structured Output is Key**: Use Pydantic and `with_structured_output` for routing and grading. It makes the code deterministic.
3. **Iterate Fast**: Use local Hugging Face chunks for blazing-fast retrieval while prototyping your RAG loops.

---